# Practice Session 08: Communities

<font size="+2" color="blue">Additional results: additional partitioning algorithm</font>

In [ ]:
# LEAVE AS-IS

import io
import networkx as nx
import matplotlib.pyplot as plt
import random
import numpy as np
import statistics

# 1. The graph

In [ ]:
# LEAVE AS-IS

INPUT_GRAPH_FILENAME = "got.graphml"

# Read the graph in GraphML format
graph_in = nx.read_graphml(INPUT_GRAPH_FILENAME)

# Re-label the nodes so they use the 'name' as label
graph_relabeled = nx.relabel.relabel_nodes(graph_in, dict(graph_in.nodes(data='name')))

# Convert the graph to undirected
graph = graph_relabeled.to_undirected()

In [ ]:
# LEAVE AS-IS (OR MODIFY VISUALLY)

def plot_graph(g, width=20, height=20, font_size=12, partition=None):

    # Create a plot of width x height
    plt.figure(figsize=(width, height))

    # By default the partition is going to be all nodes in the same partition
    if partition is None:
        partition = [ set(g.nodes()) ]
        
    # Number of partitions
    num_parts = len(partition)
    
    # Create a map from nodes to color using color values from 0.0 for the first partition
    # to 1-1/P for the last partition, assuming there are P partitions
    node_to_color = {}
    part_color = 0.0
    for part in partition:
        for node in part:
            node_to_color[node] = part_color
        part_color += 1.0/num_parts
    
    # Create a list of colors in the ordering of the nodes
    colors = [node_to_color[node] for node in g.nodes()]
    
    # Layout the nodes using a spring model
    nx.draw_spring(g, with_labels=True, node_size=1000, font_size=font_size,
                   cmap=plt.get_cmap('YlOrRd'), node_color=colors)

    # Display
    plt.show()

In [ ]:
# LEAVE AS-IS

plot_graph(graph)

# 2. K-core decomposition

In [ ]:
# LEAVE AS-IS

def get_max_degree(g):
    degree_sequence = [x[1] for x in g.degree()]
    return(max(degree_sequence))


def nodes_with_degree_less_or_equal_than(g, degree):
    nodes = []
    for node in g.nodes():
        if g.degree(node) <= degree:
            nodes.append(node)
    return nodes

In [ ]:
def kcore_decomposition(graph):
    g = graph.copy()
    max_degree = get_max_degree(g)

    node_to_level = {}
    for level in range(1, max_degree + 1):

        while True:
            # Obtain the list of nodes with degree <= level
            nodes_in_level = nodes_with_degree_less_or_equal_than(g, level)

            # Check if this list is empty
            if len(nodes_in_level) == 0:
                break # This level is finished, move to next one

            # If the list is not empty, assign the nodes to the
            # corresponding level and remove the node
            for node in nodes_in_level:
                node_to_level[node] = level # Assign the level to the node
                g.remove_node(node) # Remove node from graph so it does not repeat

    return(node_to_level)

In [ ]:
# LEAVE AS-IS

node_to_kcore = kcore_decomposition(graph)

for character in ["Jon Snow", "Tyrion Lannister", "Night King"]:
    print("K-core of {:s}: {:d}".format(character, node_to_kcore[character]))

In [ ]:
graphcore = graph.subgraph([node for node in node_to_kcore if node_to_kcore[node]>=4])

plot_graph(graphcore)

<font size="+1" color="red">We can see very clearly that there are two communities that can be split with a cut of size 1.</font>

# 3. Modularity of a partition

In [ ]:
# Leave as-is

g = nx.Graph()

g.add_edge(0, 1)
g.add_edge(1, 2)
g.add_edge(2, 3)
g.add_edge(3, 0)
g.add_edge(0, 2)
g.add_edge(3, 4)
g.add_edge(4, 5)
g.add_edge(5, 6)
g.add_edge(6, 4)

plot_graph(g, height=3, width=18, font_size=30)

In [ ]:
# LEAVE AS-IS

partition1 = [
    {0, 1, 2, 3},
    {4, 5, 6}
]
plot_graph(g, height=3, width=18, font_size=30, partition=partition1)
print("Modularity of partition 1 according to NetworkX: %.4f" % nx.community.quality.modularity(g, partition1))


partition2 = [
    {0, 1, 2},
    {3, 4, 5, 6}
]
plot_graph(g, height=3, width=18, font_size=30, partition=partition2)
print("Modularity of partition 2 according to NetworkX: %.4f" % nx.community.quality.modularity(g, partition2))

In [ ]:
def Lc(g: nx.Graph, C):
    internal_links = [0]*len(C)
    for partition, i in zip(C, range(len(C))):
        subgraph = g.subgraph(partition)
        internal_links[i] = subgraph.number_of_edges()
    return internal_links

def Kc(g: nx.Graph, C):
    sum_degrees = [0]*len(C)
    for partition, i in zip(C, range(len(C))):
        for node in partition:
            sum_degrees[i] += g.degree(node)
    return sum_degrees

def modularity(g: nx.Graph, partition):
    modularity = 0.0

    L = g.number_of_edges()
    lc = Lc(g, partition)
    kc = Kc(g, partition)

    for i in range(len(partition)):
        modularity += lc[i] - kc[i]**2/(4*L)
    return modularity/L

In [ ]:
# LEAVE AS-IS

print("Modularity of partition 1: mine={:.6f}, networkx={:.6f}".format(
    modularity(g, partition1), nx.community.quality.modularity(g, partition1)))

print("Modularity of partition 2: mine={:.6f}, networkx={:.6f}".format(
    modularity(g, partition2), nx.community.quality.modularity(g, partition2)))


# 4. Girvan-Newman algorithm

## 4.1. Find the edge with the largest betweenness

In [ ]:
def largest_betweenness_edge(g: nx.Graph):
    edge_betweenness = nx.edge_betweenness_centrality(g)
    largest = sorted(edge_betweenness.keys(), key=lambda x:edge_betweenness[x], reverse=True)[0]
    return largest[0:2]

In [ ]:
# LEAVE AS-IS
# The answer should be pretty obvious, considering the graph

print(largest_betweenness_edge(g))

## 4.2. Iteratively remove the edge with the largest betweenness

In [ ]:
# LEAVE AS-IS

def list_connected_components(g):
    return list(nx.connected_components(g))

def number_connected_components(g):
    return len(list_connected_components(g))


In [ ]:
def girvan_newman(orig: nx.Graph):

    # Copy the original graph
    g = orig.copy()

    # All of the nodes in a single partition is the first partition we create
    partition_sequence = [list_connected_components(g)]
    
    # Compute the current number of connected components (ncomp)
    ncomp = len(partition_sequence[0])
    
    # While we have not arrived to a situation where each node is a singleton
    while ncomp < g.number_of_nodes():
        
        # Find an edge to remove and remove it
        u, v = largest_betweenness_edge(g)
        g.remove_edge(u, v)
        
        # Recompute the new number of connected components (ncomp_new)
        ncomp_new = len(list_connected_components(g))
        
        # If the number of connected components has increased
        if ncomp_new > ncomp:
            
            # Add to the partition sequence the list of connected components
            partition_sequence.append(list_connected_components(g))
            
            # Update the number of connected components
            ncomp = ncomp_new

    return partition_sequence

In [ ]:
# LEAVE AS-IS

def run_girvan_newman(g):

    partitions = girvan_newman(g)
    modularity_profile = []    
    for partition in partitions:
        print("Partition %s" % (partition,) )
        m = modularity(g, partition)
        print("Modularity: %.4f" % m)
        modularity_profile.append(m)
        print()  
        
    plt.xlabel("Iteration")
    plt.ylabel("Modularity")
    plt.title("Modularity profile")
    plt.plot(modularity_profile)

In [ ]:
# LEAVE AS-IS

run_girvan_newman(g)

In [ ]:
# LEAVE AS-IS

run_girvan_newman(graph)

In [ ]:
# LEAVE AS-IS

run_girvan_newman(graphcore)

<font size="+1" color="red">Replace this cell with a brief commentary about the modularity profiles above, and which would be the partitioning that should be chosen in each case.</font>

In [ ]:
def run_girvan_newman_modularity(g):
    partitions = girvan_newman(g)
    modularities = [modularity(g, p) for p in partitions] # List of modularities in the same order as partitions
    return partitions[modularities.index(max(modularities))] # Return the partition in index of highest modularity

In [ ]:
# LEAVE AS-IS

def run_and_plot(name, g):
    partition = run_girvan_newman_modularity(g)
    print("The best partition of {:s} has modularity {:.4f} and {:d} communities".format(
        name, modularity(g, partition), len(partition)))
    plot_graph(g, partition=partition)    

In [ ]:
# LEAVE AS-IS

run_and_plot("the entire graph", graph)

In [ ]:
# LEAVE AS-IS


run_and_plot("the core of the graph", graphcore)

<font size="+1" color="red">Replace this cell with a brief commentary about what you see in these two partitionings. If you see some interesting community or communities, you can look online to check if the characters in those communities are somehow related in the series. Do you see some consistencies or inconsistencies when comparing the partitioning of the core nodes only, and the partitions in which they are placed when partitioning the entire graph?</font>

# Extra section

In [ ]:
def assign_random_community(g: nx.Graph, C: int, randseed) -> list:
    random.seed(randseed) # 2, 3
    part = [[] for _ in range(C)]
    for node in g.nodes():
        rand_community = random.randrange(C)
        part[rand_community].append(node)
    return part

def indexOf(l: list, e):
    for entry in range(len(l)):
        if e in l[entry]:
            return entry
    return -1

def label_propagation_algorithm(g: nx.Graph, C: int, iter: int, randseed) -> list[list]:
    '''
    g: Input graph
    C: number of desired communities
    iter: algorithm iterations 
    '''

    # Initialize randomly communities
    partitions = [[] for _ in range(iter)]
    partitions[0] = assign_random_community(g, C, randseed)

    # For all iterations
    for i in range(1, iter):
        partitions[i] = [[] for _ in range(C)]
        # For every node
        for node in g.nodes():
            # Count nodes in neighbor's communities
            neighbor_count = [[0, i] for i in range(C)]
            for neighbor in g.neighbors(node):
                neighbor_count[indexOf(partitions[i-1], neighbor)][0] += 1

            # Move node to majority of neighbors community
            largest_community = sorted(neighbor_count, key=lambda x:x[0], reverse=True)[0][1]

            partitions[i][largest_community].append(node)
    return partitions

def run_label_propagation(g: nx.Graph, C: int, iter: int, randseed):
    partitions = label_propagation_algorithm(g, C, iter, randseed)
    modularity_profile = []    
    for partition in partitions:
        print("Partition %s" % (partition,) )
        m = modularity(g, partition)
        print("Modularity: %.4f" % m)
        modularity_profile.append(m)
        print()  
        
    plt.xlabel("Iteration")
    plt.ylabel("Modularity")
    plt.title("Modularity profile")
    plt.plot(modularity_profile)

In [ ]:
run_label_propagation(graph, 3, 10, 8)

The issue with this kind of implementation is that the random initialization determines how the model will perform, and if the initialization is not good it will converge to values that are suboptimal. However if we have a good initialization like with the given seed, it performs a bit worse than the Girvan Newman algorithm but it converges to a fairly good modularity.

<font size="+2" color="#003300">I hereby declare that, except for the code provided by the course instructors, all of my code, report, and figures were produced by myself.</font>